Vogliamo costruire il dataset dei prodotti Amazon nel campo Electronics, intersecando:
Amazon Reviews 2023 che contiene metadati + rating/recensioni
Amazon question/answer storico McAuley

In [9]:
# Se mancano pacchetti, esegui questa cella una sola volta.
%pip install -q "datasets==3.6.0" pandas pyarrow requests tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path #per gestire meglio percorsi di file e cartelle
import ast #servirà più avanti per leggere file del QA storico
import gzip #leggere e scrivere file compressi gz
import json #lavorare con json
import shutil #utile per varie operazioni, forse in futuro
from typing import Iterable, Optional #tipi
import pandas as pd #pandas per creare dataframe
import pyarrow as pa #per definire schemi tabellari
import pyarrow.parquet as pq    #per leggere metadata dei file .parquet
import requests #per scaricare da URL
from tqdm.auto import tqdm #barra di progresso
import datasets #pacchetto datasets
from datasets import load_dataset #per caricare dataset

datasets.logging.set_verbosity_error() #riduce i messaggi stampati da datasets

c:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#configurazione
CATEGORY_2023="Electronics" #modifica qua per cambiare categoria

QA_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa/qa_Electronics.json.gz"
#per versione risposte multiple https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa/icdm/QA_Electronics.json.gz

In [4]:
PROJECT_ROOT=Path("..").resolve() if Path("..").joinpath("data").exists() else Path(".").resolve() #sceglie root
RAW_DIR=PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR=PROJECT_ROOT / "data" / "processed" / "electronics_intersection"

QA_RAW_PATH=RAW_DIR / "qa" / "qa_Electronics.json.gz"
QA_STRICT_PATH=RAW_DIR / "qa" / "qa_Electronics.strict.jsonl.gz"

ITEMS_OUT=PROCESSED_DIR / "items.parquet" #percorso finale dei prodotti
QA_OUT=PROCESSED_DIR / "qa.parquet" #percorso file domande/risposte
RATINGS_OUT=PROCESSED_DIR / "ratings.parquet" #percorso rating

RAW_DIR.mkdir(parents=True, exist_ok=True) #crea data/raw
(QA_RAW_PATH.parent).mkdir(parents=True, exist_ok=True) #crea padre data/raw/qa
PROCESSED_DIR.mkdir(parents=True, exist_ok=True) #crea data/processed/electronics_intersection

print("Project root:", PROJECT_ROOT)
print("Output:", PROCESSED_DIR)

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Output: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection


In [5]:
def download_file(url:str,output_path:Path, force:bool=False) ->Path: #scarica da url in output_path
    if output_path.exists() and not force: #non riscarica
        print(f"Già presente: {output_path}")
        return output_path
    output_path.parent.mkdir(parents=True, exist_ok=True) #se non esiste cartella di destinazione la crea

    with requests.get(url, stream=True,timeout=60)as r:
        r.raise_for_status()
        total =int(r.headers.get("content-length", 0)) #dimensione
        with open(output_path, "wb") as f, tqdm( #apre file in scrittura binaria, perchè scarichi .gz e apre barra tqdm
            total=total,
            unit="B", #in byte
            unit_scale=True,
            desc=output_path.name,
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return output_path

download_file(QA_URL, QA_RAW_PATH)

Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Electronics.json.gz


WindowsPath('C:/Users/arsid/Desktop/Tesi-ecommerce/data/raw/qa/qa_Electronics.json.gz')

In [6]:
#Convertiamo QA storico da JSONL gzip a JSON e aggiungiamo parent_asin per
#agganciarlo al dataset amazon 2023
#per fare questo l'asin storico lo usiamo come parent_asin
def iter_historical_qa(path: Path)->Iterable[dict]: #legge qa storico, restituisce dizionari python
    with gzip.open(path,"rt",encoding="utf-8", errors="replace") as f: #apre il file .gz
        for line in f:
            line =line.strip() #rimuove spazi e newline
            if line:
                yield ast.literal_eval(line) #converte riga da testo a dizionario python

qa_asins=set()
qa_rows=0


with gzip.open(QA_STRICT_PATH, "wt", encoding="utf-8") as out: #apre il file in scrittura, .jsonl.gz
    for qa_id,row in enumerate(tqdm(iter_historical_qa(QA_RAW_PATH), desc="Conversione QA")): #scorre tutti i record del qa storico
        asin=row.get("asin") #prende valore dal campo asin
        row["qa_id"] = qa_id #aggiunge id numerico alla riga qa
        row["parent_asin"] = asin #crea il campo parent_asin e ci copia l'asin

        if asin: #se asin non è vuoto, aggiunge l'asin all'insieme dei prodotti presenti nel qa
            qa_asins.add(asin)

        out.write(json.dumps(row, ensure_ascii=False) + "\n") #scrive la riga nel nuovo file jsonl
        qa_rows += 1

print("QA rows:", qa_rows)
print("Prodotti unici nel QA:", len(qa_asins))
print("File strict:", QA_STRICT_PATH)

Conversione QA: 314263it [00:12, 25388.12it/s]

QA rows: 314263
Prodotti unici nel QA: 39371
File strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Electronics.strict.jsonl.gz


In [7]:
#carichiamo metadati amazon reviews 2023 da Hugging Face (piattaforma dove è pubblicato il dataset)
meta_ds = load_dataset( #risultato salvato in meta_ds
    "McAuley-Lab/Amazon-Reviews-2023", #nome del dataset su HuggingFace
    f"raw_meta_{CATEGORY_2023}", #raw_meta_electronics per ora
    split="full", 
    trust_remote_code=True,
)
print(meta_ds)
print("Numero metadati:", len(meta_ds))

Generating full split: 100%|██████████| 1610012/1610012 [01:55<00:00, 13901.61 examples/s]


Dataset({
    features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author'],
    num_rows: 1610012
})
Numero metadati: 1610012
